# Chapter 27: Hash Maps and Prefix Sums

The programs from Chapter 27 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Grouping and Counting

`hashing_patterns.py`

In [1]:
"""Hash maps and prefix sums: remember what you've seen, so a question
about earlier items is answered in constant time."""

from collections import Counter, defaultdict


def group_anagrams(words):
    """Words with the same letters share a key: their sorted letters."""
    groups = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)
    return list(groups.values())


def first_unique(text):
    """Index of the first character that appears once, or -1."""
    counts = Counter(text)
    return next((i for i, ch in enumerate(text) if counts[ch] == 1), -1)


def longest_consecutive(nums):
    """Length of the longest run of consecutive integers, in any order,
    in O(n): only start counting at a number with no predecessor."""
    present, best = set(nums), 0
    for x in present:
        if x - 1 not in present:
            length = 1
            while x + length in present:
                length += 1
            best = max(best, length)
    return best


def prefix_sums(nums):
    """prefix[i] is the sum of the first i numbers, so the sum of
    nums[i:j] is prefix[j] - prefix[i]: any range in one subtraction."""
    prefix = [0]
    for x in nums:
        prefix.append(prefix[-1] + x)
    return prefix


def count_subarrays_with_sum(nums, k):
    """How many runs of neighbors add up to k? A run ending here sums to
    k exactly when an earlier prefix sum equals (this prefix - k)."""
    seen = Counter({0: 1})  # the empty prefix
    total = count = 0
    for x in nums:
        total += x
        count += seen[total - k]
        seen[total] += 1
    return count

## Prefix Sums Meet Hash Maps

`hashing_demo.py`

In [2]:
"""The hash-map and prefix-sum solutions on small examples."""

from hashing_patterns import (
    count_subarrays_with_sum,
    first_unique,
    group_anagrams,
    longest_consecutive,
    prefix_sums,
)

print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))
print(first_unique("swiss"), first_unique("aabb"))
print(longest_consecutive([100, 4, 200, 1, 3, 2]))
sales = [3, 1, 4, 1, 5, 9, 2, 6]
prefix = prefix_sums(sales)
print(prefix, "sum of sales[2:6]:", prefix[6] - prefix[2])
print(count_subarrays_with_sum([1, 2, 3, -3, 1, 2], 3))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]
1 -1
4
[0, 3, 4, 8, 9, 14, 23, 25, 31] sum of sales[2:6]: 19
6


## Prefix Sums in Two Dimensions

`grid_sums.py`

In [3]:
"""Prefix sums in two dimensions: the sum of any rectangle of a grid in
four lookups, after one pass to build the table."""

grid = [
    [3, 0, 1, 4],
    [5, 6, 3, 2],
    [1, 2, 0, 1],
]
rows, cols = len(grid), len(grid[0])
# table[r][c]: the sum of the rectangle from (0, 0) up to (r-1, c-1)
table = [[0] * (cols + 1) for _ in range(rows + 1)]
for r in range(rows):
    for c in range(cols):
        table[r + 1][c + 1] = (
            grid[r][c] + table[r][c + 1] + table[r + 1][c] - table[r][c]
        )


def rect_sum(r1, c1, r2, c2):
    """Sum of rows r1..r2 and columns c1..c2, inclusive."""
    return (
        table[r2 + 1][c2 + 1]
        - table[r1][c2 + 1]
        - table[r2 + 1][c1]
        + table[r1][c1]
    )


print(rect_sum(0, 0, 2, 3), rect_sum(1, 1, 2, 2), rect_sum(0, 3, 1, 3))

28 11 6


## Timing

`bench_subarrays.py`

In [4]:
"""Counting runs with sum k: every run (O(n^2) with running sums)
against prefix sums in a hash map (O(n))."""

import random

import bench
from hashing_patterns import count_subarrays_with_sum

rng = random.Random(2026)


def brute(nums, k):
    count = 0
    for i in range(len(nums)):
        total = 0
        for j in range(i, len(nums)):
            total += nums[j]
            count += total == k
    return count


def make(n):
    return [rng.randint(-5, 5) for _ in range(n)], 3


sizes = [500, 1_000, 2_000, 4_000]
results = {
    "every run": bench.doubling(brute, make, sizes),
    "prefix + hash": bench.doubling(count_subarrays_with_sum, make, sizes),
}
bench.table(results)

         n       every run   prefix + hash
       500         3.72 ms         68.8 us
     1,000    15.4 ms x4.1     138 us x2.0
     2,000    72.3 ms x4.7     217 us x1.6
     4,000     301 ms x4.2     551 us x2.5
